# 🧪 Lab 4: The Baggage Null Torture Test (Structural Mutation & Null Discrimination Diagnostics)

In this laboratory session, we construct an aggressive structural mutation grid to evaluate how Apache Spark 4 isolates conflicting null identities, optional fields, and type drift within a single processing pipeline. We will execute a dual-track audit comparing a rigid explicit schema via `from_json` against flexible native `VARIANT` pathways. Through this process, we will build a comprehensive data quality classification framework that isolates structural anomalies safely without halting active cluster threads.

### Step 1: Initialize Local Spark Workspace
We spin up our local standalone Spark session to establish our analytical workspace environment.

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.getOrCreate()
print(f"Active Spark Engine Version: {spark.version}")

Active Spark Engine Version: 4.1.2


### Step 2: Build the Structural Mutation Grid
We assemble an 8-row mutation matrix where each record outlines a baseline flight offer, but the internal `baggageAllowance` branch alters its structural costume.

In [7]:
mutation_grid = [
    ("BKG-MISSING", '{"origin":"MAD","destination":"JFK","price":450.00}'),
    ("BKG-EXPLICIT-NULL", '{"origin":"MAD","destination":"JFK","price":450.00,"baggageAllowance":null}'),
    ("BKG-EMPTY-ARRAY", '{"origin":"MAD","destination":"JFK","price":450.00,"baggageAllowance":[]}'),
    ("BKG-SINGLE-OBJ", '{"origin":"MAD","destination":"JFK","price":450.00,"baggageAllowance":{"quantity":1,"unit":"PC"}}'),
    ("BKG-ARRAY-OBJ", '{"origin":"MAD","destination":"JFK","price":450.00,"baggageAllowance":[{"quantity":1,"unit":"PC"},{"quantity":1,"unit":"23KG"}]}'),
    ("BKG-SCALAR-STR", '{"origin":"MAD","destination":"JFK","price":450.00,"baggageAllowance":"1PC"}'),
    ("BKG-INVALID-QUANT", '{"origin":"MAD","destination":"JFK","price":450.00,"baggageAllowance":{"quantity":"one","unit":"PC"}}'),
    ("BKG-UNKNOWN-UNIT", '{"origin":"MAD","destination":"JFK","price":450.00,"baggageAllowance":{"quantity":1,"unit":"XYZ"}}')
]

bronze_df = spark.createDataFrame(mutation_grid, ["booking_id", "raw_payload"])
print("=== MUTATION GRID PREPARED ===")
bronze_df.show(truncate=False)

=== MUTATION GRID PREPARED ===
+-----------------+--------------------------------------------------------------------------------------------------------------------------------+
|booking_id       |raw_payload                                                                                                                     |
+-----------------+--------------------------------------------------------------------------------------------------------------------------------+
|BKG-MISSING      |{"origin":"MAD","destination":"JFK","price":450.00}                                                                             |
|BKG-EXPLICIT-NULL|{"origin":"MAD","destination":"JFK","price":450.00,"baggageAllowance":null}                                                     |
|BKG-EMPTY-ARRAY  |{"origin":"MAD","destination":"JFK","price":450.00,"baggageAllowance":[]}                                                       |
|BKG-SINGLE-OBJ   |{"origin":"MAD","destination":"JFK","price":450.00,"bagg

### Step 3: Track 1 — The Rigid Gate Failure Mode
We attempt to process our mutating matrix using `from_json` matched against an explicit structural assumption to witness the permissive data amnesia effect.

In [8]:
# Define an explicit schema that rigidly expects a single baggage object structure
rigid_baggage_schema = T.StructType([
    T.StructField("origin", T.StringType(), True),
    T.StructField("destination", T.StringType(), True),
    T.StructField("price", T.DoubleType(), True),
    T.StructField("baggageAllowance", T.StructType([
        T.StructField("quantity", T.IntegerType(), True),
        T.StructField("unit", T.StringType(), True)
    ]), True)
])

rigid_parsed_df = bronze_df.withColumn("parsed", F.from_json(F.col("raw_payload"), rigid_baggage_schema))

print("=== TRACK 1: RIGID SCHEMATIC OUTCOMES ===")
rigid_parsed_df.select(
    "booking_id",
    F.col("parsed.baggageAllowance.quantity").alias("quantity"),
    F.col("parsed.baggageAllowance").alias("full_baggage_struct")
).show(truncate=False)

=== TRACK 1: RIGID SCHEMATIC OUTCOMES ===
+-----------------+--------+-------------------+
|booking_id       |quantity|full_baggage_struct|
+-----------------+--------+-------------------+
|BKG-MISSING      |NULL    |NULL               |
|BKG-EXPLICIT-NULL|NULL    |NULL               |
|BKG-EMPTY-ARRAY  |NULL    |NULL               |
|BKG-SINGLE-OBJ   |1       |{1, PC}            |
|BKG-ARRAY-OBJ    |NULL    |NULL               |
|BKG-SCALAR-STR   |NULL    |NULL               |
|BKG-INVALID-QUANT|NULL    |{NULL, PC}         |
|BKG-UNKNOWN-UNIT |1       |{1, XYZ}           |
+-----------------+--------+-------------------+



### Step 4: Track 2 — Native VARIANT Pathway & Classification Logic
We ingest our mutation grid using `parse_json` and deploy progressive structural validation to categorize every supplier variant into explicit data metrics.

In [9]:
variant_df = bronze_df.withColumn("v", F.parse_json(F.col("raw_payload")))

# Isolate sub-nested variant branches to verify their data types via schema strings
baggage_v = F.variant_get(F.col("v"), "$.baggageAllowance", "variant")
baggage_schema_str = F.schema_of_variant(baggage_v)

# Deploy the data quality radar classification (checking complex structures first to avoid string substring traps)
diagnostic_df = variant_df.withColumn(
    "baggage_class",
    F.when(baggage_v.isNull() & ~F.is_variant_null(baggage_v), F.lit("missing_field"))
     .when(F.is_variant_null(baggage_v), F.lit("explicit_variant_null"))
     .when(baggage_schema_str.contains("ARRAY") & (F.variant_get(baggage_v, "$[0]", "string").isNull()), F.lit("empty_collection"))
     .when(baggage_schema_str.contains("ARRAY"), F.lit("valid_array"))
     .when(baggage_schema_str.contains("OBJECT") & F.try_variant_get(baggage_v, "$.quantity", "int").isNull(), F.lit("invalid_quantity"))
     .when(baggage_schema_str.contains("OBJECT") & ~F.variant_get(baggage_v, "$.unit", "string").isin("PC", "KG"), F.lit("unknown_unit"))
     .when(baggage_schema_str.contains("OBJECT"), F.lit("valid_single_object"))
     .when(baggage_schema_str.contains("STRING"), F.lit("invalid_scalar"))
     .otherwise(F.lit("unclassified_mutation"))
)

print("=== TRACK 2: VARIANT STRUCTURAL CLASSIFICATION REPORT ===")
diagnostic_df.select("booking_id", "baggage_class").show(truncate=False)

=== TRACK 2: VARIANT STRUCTURAL CLASSIFICATION REPORT ===
+-----------------+---------------------+
|booking_id       |baggage_class        |
+-----------------+---------------------+
|BKG-MISSING      |missing_field        |
|BKG-EXPLICIT-NULL|explicit_variant_null|
|BKG-EMPTY-ARRAY  |empty_collection     |
|BKG-SINGLE-OBJ   |valid_single_object  |
|BKG-ARRAY-OBJ    |valid_array          |
|BKG-SCALAR-STR   |invalid_scalar       |
|BKG-INVALID-QUANT|invalid_quantity     |
|BKG-UNKNOWN-UNIT |unknown_unit         |
+-----------------+---------------------+



### Step 5: Safe Production Promotion
We leverage our diagnostic classifications to compute cleanly typed serving variables, decoupling ingestion survival from downstream business rules.

In [10]:
serving_layer_df = diagnostic_df.withColumn(
    "baggage_status",
    F.when(F.col("baggage_class").isin("valid_single_object", "valid_array"), F.lit("included"))
     .when(F.col("baggage_class") == "empty_collection", F.lit("excluded"))
     .when(F.col("baggage_class") == "explicit_variant_null", F.lit("not_applicable"))
     .when(F.col("baggage_class") == "missing_field", F.lit("supplier_not_provided"))
     .otherwise(F.lit("invalid_payload"))
).withColumn(
    "checked_bag_count",
    F.when(F.col("baggage_class") == "valid_single_object", F.variant_get(baggage_v, "$.quantity", "int"))
     .when(F.col("baggage_class") == "valid_array", F.variant_get(baggage_v, "$[0].quantity", "int"))
     .when(F.col("baggage_class") == "empty_collection", F.lit(0))
     .otherwise(F.lit(None).cast("int"))
).withColumn(
    "baggage_parse_status",
    F.when(F.col("baggage_status") == "invalid_payload", F.lit("FAIL")).otherwise(F.lit("SUCCESS"))
).withColumn(
    "baggage_error_reason",
    F.when(F.col("baggage_class") == "invalid_scalar", F.lit("Baggage arrived as primitive text string"))
     .when(F.col("baggage_class") == "invalid_quantity", F.lit("Alphanumeric value passed under object quantity metric"))
     .when(F.col("baggage_class") == "unknown_unit", F.lit("Unrecognized weight/piece measurement unit code"))
     .otherwise(F.lit(None).cast("string"))
)

print("=== FINAL SANITIZED PROMOTED LAYER ===")
serving_layer_df.select(
    "booking_id",
    "baggage_status",
    "checked_bag_count",
    "baggage_parse_status",
    "baggage_error_reason"
).show(truncate=False)

=== FINAL SANITIZED PROMOTED LAYER ===
+-----------------+---------------------+-----------------+--------------------+------------------------------------------------------+
|booking_id       |baggage_status       |checked_bag_count|baggage_parse_status|baggage_error_reason                                  |
+-----------------+---------------------+-----------------+--------------------+------------------------------------------------------+
|BKG-MISSING      |supplier_not_provided|NULL             |SUCCESS             |NULL                                                  |
|BKG-EXPLICIT-NULL|not_applicable       |NULL             |SUCCESS             |NULL                                                  |
|BKG-EMPTY-ARRAY  |excluded             |0                |SUCCESS             |NULL                                                  |
|BKG-SINGLE-OBJ   |included             |1                |SUCCESS             |NULL                                                  |
|BKG-ARRA

## 📊 Post-Lab Analysis: Evidence from the Engine Room

### 1. Complex Type Isolation & The Substring Reordering Fix
* **Defeating the Token Trap:** Reviewing the classification report in Step 4 confirms that complex structures are now cleanly isolated. By placing the `OBJECT` and `ARRAY` string containment checks upstream of the bare primitive `STRING` check, rows like `BKG-SINGLE-OBJ`, `BKG-INVALID-QUANT`, and `BKG-UNKNOWN-UNIT` successfully route to their exact type metrics rather than collapsing into false positives.
* **True Progressive Promotion:** The serving layer printout under Step 5 documents the analytical reward of this structural parsing order. Valid single objects and multiple-object arrays are promoted smoothly to an `included` status, while the text-wrapped integer and unknown token strings are caught by the type-drift filters and converted into contextual error logs without crashing active worker threads.

### 2. Rigid Schematic Total Amnesia Verified
* **The Struct Destruction Filter:** The Track 1 outputs under Step 3 expose the exact structural penalty tax of old-school parsing gates. When the array structure of `BKG-ARRAY-OBJ` or the loose text string of `BKG-SCALAR-STR` hit the rigid assumptions of `from_json`, the engine room wiped out the entire nested branch, defaulting the columns to a silent, un-logged SQL `NULL` and erasing all historical data evidence.
* **Localized Cell Decay:** Conversely, the output for `BKG-INVALID-QUANT` reveals an interesting edge case where the structural shell matches but the inner leaf-node data shifts. Because the incoming payload maintained an object wrapper with a `quantity` key, the rigid parser preserved the outer container, selectively destroying only the text-wrapped `"one"` into a local SQL `NULL` while successfully retaining the `unit: "PC"` token.

### 3. Flat Infrastructure Tax & Catalyst Coordination Latency
* **Uniform Ingestion Clocks:** Evaluating the raw execution timers across your cells highlights a highly flat latency signature, with Step 2 executing in 13.01 seconds, Step 3 in 13.13 seconds, Step 4 in 11.45 seconds, and Step 5 finishing in 11.94 seconds.
* **Coordination vs. Execution Overheads:** Because the processing logic handles an identical micro-dataset of eight rows across all passes, this execution time is completely decoupled from actual data scale or parsing computational intensity. The timestamps confirm that your processing runway is paying a fixed local infrastructure tax for Catalyst logical optimization, DataFrame graph compilation, and JVM worker thread task serialization loops.